# Aegis Alpha - Phase 5: End-to-End Automated Workflow Runner

This notebook imports and executes all modular Python pipeline modules (`src/data_fetcher.py`, `src/features.py`, `src/train.py`, `src/simulator.py`, `src/broker.py`, `src/notifier.py`) end-to-end and outputs the latest predicted trades.

Set `RUN_MODE = 'offline'` for instant peer-review reproduction from a pre-saved SQLite database or Parquet snapshot, or `RUN_MODE = 'online'` to perform live incremental API ingestion from Binance and Alternative.me.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import DB_PATH, DATA_DIR, MODELS_DIR, SYMBOLS, configure_logging
from src.features import get_feature_catalog
from src.simulator import compute_btc_benchmark
from main import run_pipeline

configure_logging('INFO')

# Toggle between 'offline' (fast peer-review reproduction) and 'online' (live API sync)
RUN_MODE = 'offline'
FORCE_RETRAIN = False
SKIP_LSTM = False
EXECUTE_ORDERS = False  # Keep False for safe Dry-Run mode

catalog = get_feature_catalog()
print('Feature Contract Summary:')
for k, v in catalog.items():
    print(f'  {k:<15}: {len(v)} columns')

## 1. Execute Complete Pipeline (`main.run_pipeline`)

In [ ]:
result = run_pipeline(
    mode=RUN_MODE,
    db_path=DB_PATH,
    offline_path=DATA_DIR / 'offline_snapshot.parquet',
    models_dir=MODELS_DIR,
    retrain=FORCE_RETRAIN,
    skip_lstm=SKIP_LSTM,
    tuning_method='optuna',
    tuning_trials=12,
    lstm_epochs=6,
    execute_orders=EXECUTE_ORDERS,
    notify=True,
)
print(f"Pipeline completed in mode={result['mode']} | eval_rows={result['eval_rows']:,}")

## 2. Multi-Model Validation and Test Comparison

In [ ]:
model_metrics = result.get('model_metrics', {})
if 'test' in model_metrics:
    test_rows = []
    for name, m in model_metrics['test'].items():
        row = {'model_or_rule': name}
        row.update(m)
        test_rows.append(row)
    display(pd.DataFrame(test_rows).set_index('model_or_rule'))
else:
    print('No model_metadata.json metrics loaded; run with FORCE_RETRAIN=True to regenerate.')

## 3. Exact Portfolio Equity Curve vs. BTC Buy-and-Hold Benchmark

In [ ]:
equity_df = result['equity_df']
preds_df = result['predictions_df']
bench_eq, bench_metrics = compute_btc_benchmark(preds_df)

fig, ax = plt.subplots(figsize=(13, 5))
if not equity_df.empty:
    ax.plot(equity_df['datetime'], equity_df['equity'], label='Aegis Alpha (Exact Sim)', linewidth=2)
if not bench_eq.empty:
    ax.plot(bench_eq['datetime'], bench_eq['equity'], label='BTC/USDT Buy-and-Hold', linestyle='--', linewidth=1.8)
ax.set_title('Aegis Alpha Exact Portfolio Equity vs. BTC Buy-and-Hold ($10,000 Start)')
ax.set_xlabel('UTC Date')
ax.set_ylabel('Portfolio Equity (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Deep Diagnostic Breakdown (Ticker, Month, Sentiment Regime, False Positives)

In [ ]:
diag = result['diagnostics']
print('Diagnostics by Ticker:')
display(pd.DataFrame(diag['by_ticker']))

print('Diagnostics by Sentiment Regime:')
display(pd.DataFrame(diag['by_sentiment_regime']))

print('Diagnostics by Volatility Regime:')
display(pd.DataFrame(diag['by_volatility_regime']))

print('Diagnostics by Month:')
display(pd.DataFrame(diag['by_month']))

fp_debug = diag.get('false_positive_debugging', {})
print('False Positive Reduction Summary:')
for k, v in fp_debug.items():
    if k != 'fp_vs_tp_feature_profile':
        print(f'  {k:<32}: {v}')
if 'fp_vs_tp_feature_profile' in fp_debug:
    display(pd.DataFrame(fp_debug['fp_vs_tp_feature_profile']).T)

## 5. Latest Predicted Trades and Broker Execution Receipts

In [ ]:
latest_bar = (
    preds_df.sort_values('timestamp')
    .groupby('symbol', as_index=False)
    .tail(1)
    .sort_values('pred_prob_ensemble', ascending=False)
)
display_cols = [
    c for c in [
        'symbol', 'datetime_utc', 'close', 'rsi_14', 'fg_value',
        'sentiment_regime', 'volatility_regime', 'pred_prob_xgb',
        'pred_prob_lstm', 'pred_prob_ensemble', 'pred_baseline', 'pred_custom_rule'
    ] if c in latest_bar.columns
]
print('Latest Bar Predictions Across All Monitored Pairs:')
display(latest_bar[display_cols])

orders = result.get('orders', [])
if orders:
    print('Generated Broker Order Receipts:')
    display(pd.DataFrame(orders))
else:
    print('No trades triggered on the latest bar under pred_custom_rule filters.')